In [ ]:
import argparse
import json
import os
import sys

In [ ]:
import joblib
import numpy as np
import pandas as pd

In [ ]:
try:
    from src import (
        config,
        data_loader,
        evaluation_suite,
        models,
        real_data_validator,
    )
except ImportError:
    import config
    import data_loader
    import evaluation_suite
    import models
    import real_data_validator

In [ ]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="ECG Pipeline: Multi-Model Benchmark & Real Validation")
    parser.add_argument(
        "--validate-real",
        action="store_true",
        default=True,
        help="Validate synthetic-trained model on real MIT-BIH PhysioNet data",
    )
    parser.add_argument(
        "--no-real",
        action="store_false",
        dest="validate_real",
        help="Skip real MIT-BIH validation",
    )
    args = parser.parse_args()

    print("=" * 80)
    print("STEP 1: Generating realistic ECG dataset with clinical class imbalance...")
    print("=" * 80)
    class_ratios = {0: 0.70, 1: 0.15, 2: 0.15}
    df = data_loader.generate_mock_features(
        num_samples=1000,
        class_ratios=class_ratios,
        noise_level=1.0,
        seed=42,
    )
    feature_cols = [
        c for c in df.columns if c not in {"patient_id", "record_id", "target"}
    ]
    print(
        f"Generated dataset with {len(df)} samples and {len(feature_cols)} features across {df['patient_id'].nunique()} patients."
    )
    print("Class distribution:\n", df["target"].map(config.TARGET_MAPPING).value_counts())

    print("\n" + "=" * 80)
    print("STEP 2: Splitting records into DS1 training and DS2 test sets...")
    print("=" * 80)
    X_train, y_train, groups_train, X_test, y_test, groups_test = (
        data_loader.split_ds1_ds2(df)
    )
    print(
        f"DS1 training rows: {len(X_train)} ({groups_train.nunique()} patients); "
        f"DS2 test rows: {len(X_test)} ({groups_test.nunique()} patients)."
    )

    print("\n" + "=" * 80)
    print("STEP 3: Multi-Model Benchmark (6 Models, GroupKFold, SMOTE, RandomizedSearchCV)...")
    print("=" * 80)
    benchmark = models.compare_all_models(
        X_train=X_train,
        y_train=y_train,
        groups_train=groups_train,
        X_test=X_test,
        y_test=y_test,
        cv_splits=5,
        n_iter=8,
        random_state=42,
    )

    comparison_df = benchmark["comparison_df"]
    best_model_name = benchmark["best_model_name"]
    best_pipeline = benchmark["best_pipeline"]

    print("\n" + "=" * 80)
    print("MODEL COMPARISON TABLE (Ranked by DS2 Macro-F1)")
    print("=" * 80)
    print(comparison_df.to_string(index=False))
    comparison_df.to_csv("model_comparison.csv", index=False)
    evaluation_suite._copy_to_artifact_dir("model_comparison.csv")

    print("\n" + "=" * 80)
    print(f"STEP 4: In-depth Diagnostics on Best Model ({best_model_name})...")
    print("=" * 80)

    # 1. SHAP Feature Importance
    shap_df = evaluation_suite.run_shap_analysis(
        pipeline=best_pipeline,
        X_train=X_train,
        X_test=X_test,
        y_test=y_test,
        output_plot_path="shap_feature_importance.png",
        max_display=15,
    )
    print("\nTop 5 SHAP Features:")
    print(shap_df.head(5)[["Feature", "Mean_Abs_SHAP"]].to_string(index=False))

    # 2. Ablation Study
    ablation_df = evaluation_suite.run_ablation_study(
        X_train=X_train,
        y_train=y_train,
        groups_train=groups_train,
        X_test=X_test,
        y_test=y_test,
        output_plot_path="ablation_study.png",
    )
    print("\nAblation Study Summary:")
    print(ablation_df[["Experiment", "Num_Features", "DS2_Macro_F1", "Delta_Macro_F1"]].to_string(index=False))

    # 3. Robustness Curve
    robustness_df = evaluation_suite.run_robustness_curve(
        pipeline=best_pipeline,
        test_patient_ids=list(groups_test.unique()),
        class_ratios=class_ratios,
        noise_levels=[0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 3.0],
        output_plot_path="robustness_curve.png",
    )

    # 4. Probability Calibration
    cal_df = evaluation_suite.run_probability_calibration(
        raw_pipeline=best_pipeline,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        output_plot_path="probability_calibration.png",
    )

    # 5. Error Analysis
    error_df = evaluation_suite.run_error_analysis(
        pipeline=best_pipeline,
        X_test=X_test,
        y_test=y_test,
        groups_test=groups_test,
        output_csv_path="error_analysis.csv",
    )

    # 6. Optional Real Validation Step: PhysioNet MIT-BIH
    if args.validate_real:
        print("\n" + "=" * 80)
        print("STEP 5: Real ECG Validation (PhysioNet MIT-BIH Arrhythmia Database)...")
        print("=" * 80)
        df_real = real_data_validator.extract_real_mitdb_features(
            records=config.DS2_RECORDS,
            max_segments_per_record=25,
            save_csv_path="real_mitdb_features.csv",
        )
        df_synthetic_test = pd.DataFrame(X_test)
        df_synthetic_test["target"] = y_test.to_numpy()

        transfer_results = real_data_validator.evaluate_synthetic_to_real_transfer(
            pipeline=best_pipeline,
            df_synthetic_test=df_synthetic_test,
            df_real=df_real,
            output_prefix="domain_gap",
        )

    print("\n" + "=" * 80)
    print("STEP 6: Exporting Final Artifacts & Predictions...")
    print("=" * 80)
    joblib.dump(best_pipeline, "best_rf_model.pkl")
    evaluation_suite._copy_to_artifact_dir("best_rf_model.pkl")

    y_pred_best = best_pipeline.predict(X_test)
    y_proba_best = best_pipeline.predict_proba(X_test)

    pred_data = {
        "patient_id": groups_test.to_numpy(),
        "y_true": y_test.to_numpy(),
        "y_pred": y_pred_best,
    }
    for idx, c in enumerate(best_pipeline.classes_):
        pred_data[f"probability_{config.TARGET_MAPPING[c]}"] = y_proba_best[:, idx]

    df_preds = pd.DataFrame(pred_data, index=y_test.index)
    df_preds.to_csv("ds2_predictions.csv", index=False)
    evaluation_suite._copy_to_artifact_dir("ds2_predictions.csv")

    print(f"Pipeline complete. Best model: {best_model_name} (Macro-F1 on DS2: {comparison_df.iloc[0]['DS2 Macro-F1']:.4f}).")
    if args.validate_real:
        print(f"Real MIT-BIH Transfer Macro-F1: {transfer_results['real_macro_f1']:.4f} (Accuracy: {transfer_results['real_accuracy']:.4f}).")